In [1]:
include("main.jl")

ventana_llegada_pasajeros_1!

# Conjuntos 

In [42]:
longitud_total_linea = 445.0
n_zonas = 3
g = 20.0
n_trenes,n_nodos = 6,10
n_pasajeros = 4

# ==========================================================
# BIG M
# ==========================================================

M = 1e4
#tiemoi cambio de aguja
varsigma = 3.0
#presupuesto disponible en MUSD
presupuesto = 100.0
#escala peso para construccion de laderos
beta = 100

E = 60*24 #minutos del dia

v_carga, v_pasajeros = 50.0,70.0 #velocidad en km/h

W_pasajero, W_carga = 15.0, 1.0 #peso en retrasar horarios
tolerancia_salida = 30 #min
tolerancia_llegada = 120 #min
t_max_ladero = 40 #min

40

# Parametros

In [43]:
Q,Qe,Qc,P,N = crear_instancias(n_trenes,n_nodos)
direccion = crear_diecciones(N,n_trenes)
origen, destino = crear_origen_destino(N,Q,direccion)
tipo = generar_tipos_tren(N, n_pasajeros)
N_pasajeros = [i for i in N if tipo[i] == 1]
N_carga = [i for i in N if tipo[i] == 0]

2-element Vector{Int64}:
 3
 5

In [44]:
b_plus, b_minus  = pares_mis_op_sentido(N)
Q_i, P_i = visitados_Q_P(Q,P)
Omega = conjunto_Omega(b_plus,b_minus,P_i)
ThetaSet = Conjunto_Theta(b_plus,b_minus,Q)
C = conjunto_zonas(n_zonas)
Kappa = conjunto_kappa(Qe,Qc)
Zset = conjunto_Z(Q,C)
sigma = sigma_zonas(C,longitud_total_linea);

In [45]:
#fijamos manualmente donde hay laderos o estaciones existentes
phi = Dict()
phi[1] = 0.0
phi[n_nodos] = longitud_total_linea;

In [46]:
lambda_prog,lambda_min, lambda_max = construir_lambdas(Qe, phi, 70, tolerancia_llegada)

(Dict{Any, Any}(10 => 381.4285714285714, 1 => 0.0), Dict{Any, Any}(10 => 261.4285714285714, 1 => -120.0), Dict{Any, Any}(10 => 501.4285714285714, 1 => 120.0))

# Parametros

In [47]:
U = costo_zonas(C;alpha = 5)
headway = headways(N, P, tipo;h_pasajero=5.0,h_mixto=10.0,h_carga=15.0)
v_T = v_trenes(N,tipo ;v_carga,v_pasajeros)
#longitud_total = 445.0
#longitud_segmento = generar_longitudes(P, longitud_total)
#tiempo_segmento = tiempo_segemento(N,P,longitud_segmento,v_T)
tau = generar_tau(N,Q,Qe,tipo)
f =  generar_f(N, tipo)
t_ladero = generar_t_ladero(N,Q)
#zonas = generar_zonas(Qc)
#costo_ladero =generar_costos_ladero(Qc, zonas)
W = generar_pesos(N, tipo;W_pasajero ,W_carga )
horario_salida = generar_horario_salida(N, direccion; intervalo = 20.0);

# Vaiables del modelo

In [48]:
model = Model(HiGHS.Optimizer)
set_attribute(model, "time_limit", 15*60.0)
set_attribute(model, "mip_rel_gap", 0.01)

In [49]:
A,D,o,retraso,d = variables_1!(model,N,Q)
z,x,theta = variables_2!(model,Zset,Omega,ThetaSet);

# Funcion Objetivo

In [50]:
funcion_objetivo!(model, N,Q,origen,horario_salida,W,retraso,A, D,z, Zset, U, beta);

# Restriccion llegada-salida

In [51]:
ventana_salida!(model, N, origen, horario_salida, D,tolerancia_salida);

In [52]:
#ventana_llegada_pasajeros_1!(model, N_pasajeros, destino, lambda_prog, lambda_max, A)

# Restriccion infaestructura

In [53]:
nodos_existentes!( model,Qe, d, phi);

In [54]:
una_zona_por_ladero!(model, Qc,C, z);

In [55]:
laderos_existentes!(model, Qe, C, z);

In [56]:
activacion_ladero!( model, N, Q, C, M, o, z);

In [57]:
zonas_construccion!(model, Kappa, C,sigma, M, d,z)

In [58]:
separacion_minima!( model, Qc, C, d,z, g,M)

In [59]:
horizonte_tiempo!( model, N,destino, A, E);

# Restriccion de Presupuesto

In [60]:
presupuesto!(model,Qc, C,U,z,presupuesto)

5 _[199] + 10 _[200] + 15 _[201] + 5 _[202] + 10 _[203] + 15 _[204] + 5 _[205] + 10 _[206] + 15 _[207] + 5 _[208] + 10 _[209] + 15 _[210] + 5 _[211] + 10 _[212] + 15 _[213] + 5 _[214] + 10 _[215] + 15 _[216] + 5 _[217] + 10 _[218] + 15 _[219] + 5 _[220] + 10 _[221] + 15 _[222] <= 100

# Restricciones de Movimiento

In [61]:
movimiento_dp!( model,N, P,direccion,A, D, d, v_T)

In [62]:
restriccion_llegada_salida!( model,  N, Q, A,D);

In [63]:
restriccion_activacion_meet_pass!(model, N,Q, M,tau, A,D, o)
restriccion_tiempo_ladero!(model, N, Q,tau,f, t_ladero, varsigma,A, D, o);

In [64]:
#espera_maxima_ladero!(model, N_carga, Q, A, D, tau, o, M, t_max_ladero)
espera_maxima_ladero!(model, N, Q, A, D, tau, o, M, t_max_ladero);

# Restricciones de conflicto

In [65]:
conflicto_mismo_sentido_salida_1!(model,  b_plus,   P, direccion, M, headway, varsigma, D, o,x)

In [66]:
conflicto_mismo_sentido_llegada_1!( model,  b_plus,  P, direccion, M,headway, varsigma, A, o, x)

In [67]:
conflicto_opuesto_1!( model, b_minus,P, direccion, M, headway, varsigma, A, D, x)

In [68]:
#espejo de las anteiores tres
conflicto_mismo_sentido_salida_2!( model, b_plus, P, direccion, M, headway,varsigma,  D, o, x)

In [69]:
conflicto_mismo_sentido_llegada_2!( model, b_plus, P, direccion,  M, headway, varsigma, A, o,x)

In [70]:
conflicto_opuesto_2!( model, b_minus,  P, direccion,  M,  headway, varsigma, A, D, x)

# Restricciones de laderos

In [71]:
restriccion_theta_1!( model, ThetaSet,P, o,x, theta)

In [72]:
restriccion_theta_2!( model, ThetaSet,  P, o, x, theta)

In [73]:
capacidad_ladero!( model, ThetaSet, P,M, varsigma,headway, A,  D, theta)

In [74]:
println(num_variables(model))
println(JuMP.num_constraints(model; count_variable_in_set_constraints = false))

510
1596


In [ ]:
optimize!(model)

println(termination_status(model))
println(primal_status(model))

In [ ]:
imprimir_laderos_construidos(Q, C, z, d)

In [ ]:
imprimir_segmentos(P, d)

In [ ]:
imprimir_nodos(Q, d)

In [ ]:
plot_trenes_km( N, Q, A, D, d,direccion)

In [ ]:
plot_trenes_nodos(N,Q, A, D,direccion)